# StoreDNA — Pipeline stages (vertical)

Data flows **top to bottom**. This notebook implements **Stage 2** (highlighted).

```
┌─────────────────────────────────────────────────────────────┐
│  1. Source Data                                             │
│  Excel workbooks — store master, ops, reviews, reports,     │
│  news, products, images                                     │
└─────────────────────────────────────────────────────────────┘
                              ↓
┌─────────────────────────────────────────────────────────────┐
│  2. Curate Data — in Cursor  ◀── YOU ARE HERE               │
│  Validate store_id · dim_store · facts · staging tables     │
│  reviews · reports · news · products                        │
└─────────────────────────────────────────────────────────────┘
                              ↓
┌─────────────────────────────────────────────────────────────┐
│  3. AI Enrichment — Azure OpenAI                            │
│  text-embedding-3-large + GPT-4.1                           │
│  Embed reviews, reports, news, products                     │
└─────────────────────────────────────────────────────────────┘
                              ↓
┌─────────────────────────────────────────────────────────────┐
│  4. Modality Vectors                                        │
│  structured · review · report · news · product · vision     │
└─────────────────────────────────────────────────────────────┘
                              ↓
┌─────────────────────────────────────────────────────────────┐
│  5. StoreDNA Builder                                        │
│  Late fusion → store_dna_vector (store fingerprint)         │
└─────────────────────────────────────────────────────────────┘
                              ↓
┌─────────────────────────────────────────────────────────────┐
│  6. Vector Index — Azure AI Search                          │
│  Peer store lookup · clustering · similarity search         │
└─────────────────────────────────────────────────────────────┘
                              ↓
┌─────────────────────────────────────────────────────────────┐
│  7. Business Output                                         │
│  Peer stores · clusters · recommendations                   │
│  GPT explains results in plain language                     │
└─────────────────────────────────────────────────────────────┘
```

| Stage | Name | Where | Status |
|-------|------|-------|--------|
| 1 | Source Data | Excel in `data/USA_100_Stores/` | Complete |
| **2** | **Curate Data** | **This notebook (Cursor)** | **Current** |
| 3 | AI Enrichment | Azure OpenAI | Next |
| 4 | Modality Vectors | Embeddings per signal | Next |
| 5 | StoreDNA Builder | Late fusion | Next |
| 6 | Vector Index | Azure AI Search | Next |
| 7 | Business Output | Apps / GPT narratives | Next |


# Retail Store DNA Builder

**Stage 2 — Curate Data (local, in Cursor)**

This notebook implements the **Curate Data** step of the StoreDNA pipeline (see stage diagram above).

### Inputs

| File | Location | Role |
|------|----------|------|
| `scraped_100_reviews_news1.xlsx` | `data/USA_100_Stores/` | **Real** scraped reviews & local news |
| `Syenthetic_data_100stores_operations_products.xlsx` | `data/USA_100_Stores/` | **Synthetic** ops & products |

### Outputs

Curated tables written to `data/USA_100_Stores/curated/`:

| Table | Type | Downstream use (Stage 3+) |
|-------|------|---------------------------|
| `dim_store` | Dimension | Join key for all modalities |
| `stg_reviews` | Staging | Review text embeddings |
| `stg_news` | Staging | News text embeddings |
| `stg_reports` | Staging | Operational report embeddings |
| `stg_products` | Staging | Product description embeddings |
| `fact_operations_weekly` | Fact | Structured ops signals / structured modality |
| `meta/scrape_run_log.csv` | Audit only | Scrape QA — not modeled |
| `meta/data_dictionary.csv` | Reference only | Field definitions — not modeled |

## Curation plan

### End objective

Produce **clean, validated, store-aligned tables** for 100 US stores (`USR-001` … `USR-100`) so that:

1. Every row references a valid `store_id` in `dim_store`.
2. Text tables (`reviews`, `news`, `reports`, `products`) are deduplicated and ready for **Azure OpenAI embedding** (Stage 3).
3. Structured ops weekly metrics are typed and keyed by `store_id` + `week_end_date`.
4. Metadata sheets (`data_dictionary`, `scrape_run_log`) are **extracted separately** and excluded from modeling.

### What we read

**Workbook A — scraped reviews & news**

| Sheet | Action |
|-------|--------|
| `store_catalog` | Merge into `dim_store` (authoritative store list) |
| `customer_reviews` | Curate → `stg_reviews` |
| `local_news` | Curate → `stg_news` |
| `scrape_run_log` | **Skip modeling** → `meta/scrape_run_log.csv` |

**Workbook B — synthetic operations & products**

| Sheet | Action |
|-------|--------|
| `data_dictionary` | **Skip modeling** → `meta/data_dictionary.csv` |
| `store_catalog` | Cross-check against `dim_store` |
| `operations_weekly` | Curate → `fact_operations_weekly` |
| `operational_reports` | Curate → `stg_reports` |
| `product_descriptions` | Curate → `stg_products` |

### Curation steps (in order)

1. **Load** both Excel workbooks into memory.
2. **Extract meta sheets** — save dictionary & scrape log to `curated/meta/`.
3. **Build `dim_store`** — one row per store; validate scraped vs synthetic catalogs match.
4. **Validate foreign keys** — drop rows with unknown `store_id`.
5. **Normalize schemas** — standard column order; strip denormalized `retailer` / `city` from synthetic facts (already on `dim_store`).
6. **Clean text tables** — trim text; reviews: keep all sources per store, reassign unique IDs; news: dedupe duplicate headlines per store.
7. **Type coercion** — dates, numeric ops fields, boolean `in_stock`.
8. **Write outputs** — CSV per table + `curation_manifest.json` with row counts and QC flags.

### What we deliberately do *not* do here

- No Azure ingest, raw zone, or Databricks feature engineering (deferred until production).
- No embeddings or StoreDNA fusion (Stages 3–7) — those follow in later notebooks.


## 1. Setup

In [ ]:
%pip install -q -r ../requirements.txt

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.store_dna_curator import (
    DIM_STORE_COLS,
    SCRAPED_META_SHEETS,
    SCRAPED_MODEL_SHEETS,
    SYNTHETIC_META_SHEETS,
    SYNTHETIC_MODEL_SHEETS,
    build_dim_store,
    curate_news,
    curate_operations_weekly,
    curate_products,
    curate_reports,
    curate_reviews,
    load_workbook_sheets,
    run_curation,
)

DATA_DIR = PROJECT_ROOT / "data" / "USA_100_Stores"
CURATED_DIR = DATA_DIR / "curated"
META_DIR = CURATED_DIR / "meta"

SCRAPED_XLSX = DATA_DIR / "scraped_100_reviews_news1.xlsx"
SYNTHETIC_XLSX = DATA_DIR / "Syenthetic_data_100stores_operations_products.xlsx"

print("Project root:", PROJECT_ROOT)
print("Scraped workbook:", SCRAPED_XLSX)
print("Synthetic workbook:", SYNTHETIC_XLSX)
print("Curated output:", CURATED_DIR)

Project root: d:\AICOE\Retail-StoreDNA
Scraped workbook: d:\AICOE\Retail-StoreDNA\data\USA_100_Stores\scraped_100_reviews_news1.xlsx
Synthetic workbook: d:\AICOE\Retail-StoreDNA\data\USA_100_Stores\Syenthetic_data_100stores_operations_products.xlsx
Curated output: d:\AICOE\Retail-StoreDNA\data\USA_100_Stores\curated


## 2. Inspect source workbooks

Confirm sheets exist and row counts look reasonable before curation.

In [2]:
def summarize_workbook(path: Path, model_sheets: tuple[str, ...], meta_sheets: tuple[str, ...]) -> pd.DataFrame:
    """List sheets, row counts, and whether each is modeled or meta-only."""
    sheets = load_workbook_sheets(path)
    rows = []
    for name, df in sheets.items():
        if name in meta_sheets:
            role = "meta (audit / reference)"
        elif name in model_sheets:
            role = "model"
        else:
            role = "unknown — review"
        rows.append({
            "workbook": path.name,
            "sheet": name,
            "rows": len(df),
            "columns": len(df.columns),
            "role": role,
        })
    return pd.DataFrame(rows)

inventory = pd.concat([
    summarize_workbook(SCRAPED_XLSX, SCRAPED_MODEL_SHEETS, SCRAPED_META_SHEETS),
    summarize_workbook(SYNTHETIC_XLSX, SYNTHETIC_MODEL_SHEETS, SYNTHETIC_META_SHEETS),
], ignore_index=True)

inventory

,workbook,sheet,rows,columns,role
0,scraped_100_reviews_news1.xlsx,store_catalog,100,7,model
1,scraped_100_reviews_news1.xlsx,customer_reviews,1319,8,model
2,scraped_100_reviews_news1.xlsx,local_news,1767,10,model
3,scraped_100_reviews_news1.xlsx,scrape_run_log,630,10,meta (audit / reference)
4,Syenthetic_data_100stores_operations_products....,data_dictionary,55,8,meta (audit / reference)
5,Syenthetic_data_100stores_operations_products....,store_catalog,100,7,model
6,Syenthetic_data_100stores_operations_products....,operations_weekly,5200,12,model
7,Syenthetic_data_100stores_operations_products....,operational_reports,891,14,model
8,Syenthetic_data_100stores_operations_products....,product_descriptions,1630,21,model


## 3. Load source data

In [3]:
scraped = load_workbook_sheets(SCRAPED_XLSX)
synthetic = load_workbook_sheets(SYNTHETIC_XLSX)

print("Scraped sheets:", list(scraped.keys()))
print("Synthetic sheets:", list(synthetic.keys()))

Scraped sheets: ['store_catalog', 'customer_reviews', 'local_news', 'scrape_run_log']
Synthetic sheets: ['data_dictionary', 'store_catalog', 'operations_weekly', 'operational_reports', 'product_descriptions']


## 4. Extract metadata sheets (not modeled)

These sheets support **documentation** and **scrape QA** only. They are saved under `curated/meta/` and excluded from Stages 3–7.

In [4]:
META_DIR.mkdir(parents=True, exist_ok=True)

meta_tables = {}

if "scrape_run_log" in scraped:
    meta_tables["scrape_run_log"] = scraped["scrape_run_log"]
    meta_tables["scrape_run_log"].to_csv(META_DIR / "scrape_run_log.csv", index=False)

if "data_dictionary" in synthetic:
    meta_tables["data_dictionary"] = synthetic["data_dictionary"]
    meta_tables["data_dictionary"].to_csv(META_DIR / "data_dictionary.csv", index=False)

for name, df in meta_tables.items():
    print(f"meta/{name}.csv — {len(df):,} rows")

meta/scrape_run_log.csv — 630 rows
meta/data_dictionary.csv — 55 rows


## 5. Build `dim_store` (store dimension)

Single source of truth for **100 stores**. 

In [5]:
dim_store, dim_qc = build_dim_store(
    scraped["store_catalog"],
    synthetic["store_catalog"],
)

valid_store_ids = set(dim_store["store_id"])

print(f"Stores in dim_store: {len(dim_store)}")
print("QC:", json.dumps(dim_qc, indent=2))
dim_store.head()

Stores in dim_store: 100
QC: {
  "store_count": 100,
  "only_in_scraped": 0,
  "only_in_synthetic": 0,
  "column_mismatches": []
}


,store_id,retailer,banner,store_name,city,state,store_format
0,USR-001,Walmart,Walmart,Walmart New York,New York,NY,Supercenter
1,USR-002,Target,Target,Target Los Angeles,Los Angeles,CA,Neighborhood
2,USR-003,Kroger,Kroger,Kroger Chicago,Chicago,IL,Neighborhood
3,USR-004,Costco,Costco,Costco Houston,Houston,TX,Supercenter
4,USR-005,Albertsons,Albertsons,Albertsons Phoenix,Phoenix,AZ,Neighborhood


## 6. Curate `stg_reviews` (customer reviews)

**Source:** `customer_reviews` sheet (scraped).

| Rule | Reason |
|------|--------|
| Keep valid `store_id` only | FK to `dim_store` |
| **Keep multiple rows per store** | Different `source` + different `review_text` are valid reviews |
| Do **not** dedupe on `review_id` alone | Scrape reuses `REV-USR-001-001` across sitejabber vs google_news |
| Reassign unique `review_id` | Pattern: `REV-{store_id}-{SOURCE}-{seq}` per store + source |
| Dedupe only on `store_id` + `source` + `review_text` | Exact same snippet only |
| Drop very short `review_text` | Noise / empty snippets |
| Parse `review_date` | Consistent date type |

> **Note:** `review_date` currently reflects the **scrape run date**, not the original post date on Sitejabber.

In [6]:
stg_reviews, reviews_qc = curate_reviews(scraped["customer_reviews"], valid_store_ids)

print("QC:", json.dumps(reviews_qc, indent=2))
print(stg_reviews["source"].value_counts())
stg_reviews.head(3)

QC: {
  "input_rows": 1319,
  "output_rows": 1318,
  "exact_duplicate_rows_removed": 1,
  "orphan_store_ids": 0,
  "note": "review_id reassigned per store+source; multiple rows per store kept when source or review_text differs"
}
source
sitejabber             765
google_news_reviews    553
Name: count, dtype: int64


,review_id,store_id,review_date,source,rating,sentiment,review_text,complaint_tags
0,REV-USR-001-SITE-001,USR-001,2026-06-23,sitejabber,NaN,neutral,"Walmart's reputation is mixed, with customers ...",NaN
1,REV-USR-001-SITE-002,USR-001,2026-06-23,sitejabber,NaN,positive,While some customers express loyalty and satis...,NaN
2,REV-USR-001-SITE-003,USR-001,2026-06-23,sitejabber,NaN,negative,While Walmart ultimately provided refunds in s...,NaN


## 7. Curate `stg_news` (local news)

**Source:** `local_news` sheet (scraped).

| Rule | Reason |
|------|--------|
| Dedupe on `news_id` | Primary key |
| Dedupe `store_id` + `headline` | Remove repeated RSS headlines |
| Parse `published_date` | Actual article dates from RSS |

In [7]:
stg_news, news_qc = curate_news(scraped["local_news"], valid_store_ids)

print("QC:", json.dumps(news_qc, indent=2))
stg_news.head(3)

QC: {
  "input_rows": 1767,
  "output_rows": 1767,
  "duplicate_headlines_removed": 0,
  "orphan_store_ids": 0
}


,news_id,store_id,city,state,published_date,headline,summary,event_type,demand_impact,source
0,NEWS-00001,USR-001,New York,NY,2026-04-29,"Walmart is changing its NY stores. Vestal, JC ...","Walmart is changing its NY stores. Vestal, JC ...",local_news,neutral,Press & Sun-Bulletin
1,NEWS-00002,USR-001,New York,NY,2026-05-11,Retailers Are Making Expensive Bets That Shopp...,Retailers Are Making Expensive Bets That Shopp...,local_news,neutral,The New York Times
2,NEWS-00003,USR-001,New York,NY,2026-04-24,"Walmart to remodel 23 stores in New York, incl...","Walmart to remodel 23 stores in New York, incl...",local_news,neutral,Star-Gazette


## 8. Curate `fact_operations_weekly` (structured ops)

**Source:** `operations_weekly` sheet (synthetic).

| Rule | Reason |
|------|--------|
| Drop denormalized store columns | `retailer`, `city`, etc. live on `dim_store` |
| Numeric coercion | `shrink_pct`, `oos_rate`, `fulfillment_rate`, `labor_hours` |
| Dedupe `store_id` + `week_end_date` | One row per store per week |

In [8]:
fact_ops, ops_qc = curate_operations_weekly(synthetic["operations_weekly"], valid_store_ids)

print("QC:", json.dumps(ops_qc, indent=2))
fact_ops.groupby("store_id").size().describe()

QC: {
  "output_rows": 5200,
  "stores": 100
}


count    100.0
mean      52.0
std        0.0
min       52.0
25%       52.0
50%       52.0
75%       52.0
max       52.0
dtype: float64

## 9. Curate `stg_reports` (operational reports)

**Source:** `operational_reports` sheet (synthetic).

Text field `description` will be embedded in Stage 3.

In [9]:
stg_reports, reports_qc = curate_reports(synthetic["operational_reports"], valid_store_ids)

print("QC:", json.dumps(reports_qc, indent=2))
stg_reports["issue_category"].value_counts()

QC: {
  "output_rows": 891
}


issue_category
shrink              147
inventory           129
safety              107
planogram           103
staffing             97
customer_service     90
signage              78
cleaning             72
equipment            68
Name: count, dtype: int64

## 10. Curate `stg_products` (product descriptions)

**Source:** `product_descriptions` sheet (synthetic).

| Rule | Reason |
|------|--------|
| Drop denormalized store columns | Join via `store_id` → `dim_store` |
| Dedupe on `sku_id` | Unique product per store |
| Coerce price / margin / velocity | Numeric structured fields for analytics |

In [10]:
stg_products, products_qc = curate_products(synthetic["product_descriptions"], valid_store_ids)

print("QC:", json.dumps(products_qc, indent=2))
stg_products.groupby("store_id").size().describe()

QC: {
  "output_rows": 1630,
  "stores": 100
}


count    100.000000
mean      16.300000
std        2.231546
min       10.000000
25%       15.000000
50%       16.000000
75%       18.000000
max       23.000000
dtype: float64

## 11. Validation summary

Confirm every curated table only references stores in `dim_store`.

In [11]:
def fk_check(df: pd.DataFrame, name: str) -> dict:
    orphans = df[~df["store_id"].isin(valid_store_ids)]
    return {"table": name, "rows": len(df), "orphan_rows": len(orphans)}

validation = pd.DataFrame([
    fk_check(stg_reviews, "stg_reviews"),
    fk_check(stg_news, "stg_news"),
    fk_check(fact_ops, "fact_operations_weekly"),
    fk_check(stg_reports, "stg_reports"),
    fk_check(stg_products, "stg_products"),
])

coverage = pd.DataFrame({
    "store_id": sorted(valid_store_ids),
}).merge(
    stg_reviews.groupby("store_id").size().rename("review_count"),
    on="store_id", how="left",
).merge(
    stg_news.groupby("store_id").size().rename("news_count"),
    on="store_id", how="left",
).merge(
    stg_products.groupby("store_id").size().rename("product_count"),
    on="store_id", how="left",
).fillna(0)

print("Foreign key checks:")
validation
print("\nPer-store coverage (first 10):")
coverage.head(10)

Foreign key checks:

Per-store coverage (first 10):


,store_id,review_count,news_count,product_count
0,USR-001,19,22,12
1,USR-002,16,21,15
2,USR-003,19,16,15
3,USR-004,16,16,18
4,USR-005,9,16,16
5,USR-006,17,18,16
6,USR-007,17,17,17
7,USR-008,16,18,17
8,USR-009,15,17,16
9,USR-010,16,20,16


## 12. Write curated outputs

Persist all tables to `data/USA_100_Stores/curated/` for Stage 3 (Azure OpenAI enrichment).

In [12]:
CURATED_DIR.mkdir(parents=True, exist_ok=True)

dim_store.to_csv(CURATED_DIR / "dim_store.csv", index=False)
stg_reviews.to_csv(CURATED_DIR / "stg_reviews.csv", index=False)
stg_news.to_csv(CURATED_DIR / "stg_news.csv", index=False)
fact_ops.to_csv(CURATED_DIR / "fact_operations_weekly.csv", index=False)
stg_reports.to_csv(CURATED_DIR / "stg_reports.csv", index=False)
stg_products.to_csv(CURATED_DIR / "stg_products.csv", index=False)

manifest = {
    "outputs": {
        "dim_store": len(dim_store),
        "stg_reviews": len(stg_reviews),
        "stg_news": len(stg_news),
        "fact_operations_weekly": len(fact_ops),
        "stg_reports": len(stg_reports),
        "stg_products": len(stg_products),
    },
    "quality_checks": {
        "dim_store": dim_qc,
        "stg_reviews": reviews_qc,
        "stg_news": news_qc,
        "fact_operations_weekly": ops_qc,
        "stg_reports": reports_qc,
        "stg_products": products_qc,
    },
}

(CURATED_DIR / "curation_manifest.json").write_text(
    json.dumps(manifest, indent=2), encoding="utf-8"
)

print("Wrote curated tables to:", CURATED_DIR)
print(json.dumps(manifest["outputs"], indent=2))

Wrote curated tables to: d:\AICOE\Retail-StoreDNA\data\USA_100_Stores\curated
{
  "dim_store": 100,
  "stg_reviews": 1318,
  "stg_news": 1767,
  "fact_operations_weekly": 5200,
  "stg_reports": 891,
  "stg_products": 1630
}


## 13. Next steps (Stages 3–7)

| Stage | Action | Input tables |
|-------|--------|--------------|
| **3. AI Enrichment** | Embed text with `text-embedding-3-large`; optional GPT-4.1 tags | `stg_reviews`, `stg_news`, `stg_reports`, `stg_products` |
| **4. Modality Vectors** | Pool embeddings per signal type | + structured vector from `fact_operations_weekly` |
| **5. StoreDNA Builder** | Late fusion → `store_dna_vector` | All modality vectors |
| **6. Vector Index — Azure AI Search** | Index store vectors | `store_dna_vector` |
| **7. Business Output** | Peer match, clusters, GPT narratives | Search results + `dim_store` |

---

**Curated file layout:**

```
data/USA_100_Stores/curated/
├── dim_store.csv
├── stg_reviews.csv
├── stg_news.csv
├── stg_reports.csv
├── stg_products.csv
├── fact_operations_weekly.csv
├── curation_manifest.json
└── meta/
    ├── scrape_run_log.csv
    └── data_dictionary.csv
```